# CICIoT2023 — cải thiện recall BruteForce và Web-Based

**Nguyễn Ngọc Dương: nghiên cứu tiếp theo của baseline DT/RF/XGBoost. Kaggle CPU.**

Mục tiêu thử nghiệm: recall ≥85% **từng** lớp hiếm, precision ≥30% **từng** lớp. Chọn theo recall thấp hơn của hai lớp, rồi recall trung bình, macro-F1; không hứa trước sẽ đạt mục tiêu.

**Add Input:** toàn bộ output processing, pilot và model baseline `models_20261008_073827_d29a37c3`. Chạy CPU, Restart → Run All. Kết quả `/kaggle/working/recall_experiments/<run_id>/`. Baseline cần đủ model.joblib và probabilities.npy, không chỉ bảng/ảnh.

Nếu phiên Kaggle bị ngắt: tải output/checkpoint, Add Input lại và đặt `RESUME_FROM` tới thư mục run. Các trial hoàn tất được dùng lại; trial đang fit phải chạy lại. Ngân sách 36 giờ cộng dồn qua các phiên, ledger ghi heartbeat 15 giây cho công việc bị ngắt. Không cần bật Internet nếu image trùng phiên baseline.

Notebook thực hiện A: quy tắc quyết định; B: lấy mẫu lớp đông/trọng số; C: tuning; D: SHAP feature selection nếu chưa đạt mục tiêu; ba seed; khóa tối đa hai ứng viên; xác nhận; báo cáo lại test cũ sau khóa. Có checkpoint/resume và giới hạn **36 giờ phiên CPU**, tính thời gian thực thi công việc thay vì thời gian notebook bỏ trống. Không có binary/34-class experiment; nhãn gốc chỉ để phân tầng/truy vết.

Validation/test cũ **đã được xem trong baseline**. Chia validation tìm kiếm/xác nhận giúp kiểm soát vòng nghiên cứu mới, không biến nó thành dữ liệu hoàn toàn chưa từng xem. Nếu phần xác nhận không đạt, báo kết quả, không sửa rồi mở lại như lần đầu.

Tập 1 triệu có đủ toàn bộ rare train hiện có. Mọi phương án lấy mẫu/chọn feature đều giữ các dòng rare; không sinh synthetic traffic. Bộ đánh giá dùng feature/float32 mask của baseline để so sánh, không purge riêng theo từng cấu hình. Tập rút gọn feature có thể tạo vector giao nhau mới; notebook audit và loại bộ feature đó nếu có giao nhau với evaluation.


In [ ]:
import os,sys,gc,json,time,uuid,hashlib,shutil,platform,threading,warnings,importlib.util,importlib.metadata
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib,psutil,sklearn,xgboost
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold,ParameterSampler
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier

PROCESSED_DIR=None
PILOT_DIR=None
BASELINE_DIR=None
RESUME_FROM=None
SOURCE_ID='20261007_164739_35f682b9'
PILOT_ID='pilot_20261008_022915_ea9a88b0'
BASELINE_ID='models_20261008_073827_d29a37c3'
SEED=42
MODEL_SEEDS=[42,2026,3407]
N_JOBS=min(4,os.cpu_count() or 1)
BATCH=25_000
CPU_SESSION_HOURS=36.0
MIN_PRECISION=0.30
TARGET_RECALL=0.85
RULE_GRID=[0.5,0.75,1.0,1.5,2.0,3.0,4.0,6.0]
RUN_CONFIRMATION=True
RUN_OLD_TEST_REPORT=True  # test đã được xem: chỉ báo cáo sau khi khóa, không chọn lại model.
ENABLE_SHAP_IF_NEEDED=True
VERIFY_HASHES=True
CLASS_NAMES=['Normal','BruteForce','DDoS','DoS','Mirai','Recon','Spoofing','Web-Based']
RARE=[1,7];MODELS=['DT','RF','XGBoost'];IDS=np.arange(8)
assert Path('/kaggle/input').exists() and Path('/kaggle/working').exists(), 'Chỉ chạy trên Kaggle.'
VERSIONS={k:importlib.metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','xgboost','joblib','psutil','matplotlib']}
VERSIONS['python']=platform.python_version()
try:OPTIONAL_VERSIONS={'shap':importlib.metadata.version('shap')}
except importlib.metadata.PackageNotFoundError:OPTIONAL_VERSIONS={'shap':None}
print(VERSIONS)


## 1. Định vị dữ liệu, protocol và resume
Resume cùng phiên bản thư viện/protocol. Nếu đổi protocol sau khi xác nhận/test đã mở, tạo run mới và ghi rõ lịch sử đã xem evaluation. Giới hạn thời gian được lưu xuyên phiên; công việc hoàn tất được dùng lại. Không thay source artifact.


In [ ]:
def now(): return datetime.now(timezone.utc).isoformat()
def read_json(p): return json.loads(Path(p).read_text(encoding='utf-8'))
def write_json(p,obj):
 p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
 t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8');t.replace(p)
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
 return h.hexdigest()
def oh(obj): return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def locate(explicit,filename,key,value):
 if explicit:
  p=Path(explicit);assert read_json(p/filename)[key]==value;return p
 hits=[]
 for root in [Path('/kaggle/input'),Path('/kaggle/working/processed_data'),Path('/kaggle/working/pilot_data'),Path('/kaggle/working/model_experiments')]:
  if not root.exists(): continue
  for f in root.rglob(filename):
   if any(x in f.parts for x in ['source_snapshot','source_reference','handoff','input_snapshot']): continue
   if f.parent.name.startswith('train_'): continue
   try:
    if read_json(f).get(key)==value: hits.append(f.parent.resolve())
   except (ValueError,OSError): pass
 hits=sorted(set(hits))
 if len(hits)!=1: raise ValueError(f'Chỉ rõ đường dẫn cho {value}: {hits}')
 return hits[0]
SOURCE=locate(PROCESSED_DIR,'preprocessing_config.json','run_id',SOURCE_ID)
PILOT=locate(PILOT_DIR,'pilot_manifest.json','pilot_run_id',PILOT_ID)
BASE=locate(BASELINE_DIR,'run_status.json','run_id',BASELINE_ID)
SC=read_json(SOURCE/'preprocessing_config.json');PC=read_json(PILOT/'pilot_manifest.json')
BP=read_json(BASE/'protocol.json');SCHEMA=read_json(SOURCE/'feature_schema.json')
assert read_json(BASE/'run_status.json')['status']=='complete'
assert SC['class_names']==PC['class_names']==BP['class_names']==CLASS_NAMES
FEATURES=SC['final_features'];assert FEATURES==BP['feature_names']==PC['feature_names'] and len(FEATURES)==44
assert sha(SOURCE/'preprocessing_config.json')==BP['source_config_sha256']==PC['source_config_sha256']
assert sha(PILOT/'pilot_manifest.json')==BP['pilot_manifest_sha256']
assert SC['run_id']==SOURCE_ID and PC['source_processing_run_id']==SOURCE_ID
BASE_HASHES=read_json(BASE/'artifact_checksums.json')
if VERIFY_HASHES:
 for f in ['protocol.json','val_input_audit.json','test_input_audit.json','val_float32_purge_audit.json','test_float32_purge_audit.json']:
  assert sha(BASE/f)==BASE_HASHES[f],f
assert BP['versions']=={**BP['versions'],**VERSIONS}, 'Nạp model baseline yêu cầu đúng phiên bản đã ghi; chọn Kaggle image phù hợp.'
BASE_PARAMS=BP['params']
SAMPLES={'original':None,'moderate':{2:300_000,3:100_000},'strong':{2:100_000,3:50_000}}
PROTOCOL=dict(revision='1.0',source=SOURCE_ID,pilot=PILOT_ID,baseline=BASELINE_ID,
 versions=VERSIONS,optional_versions=OPTIONAL_VERSIONS,pilot_manifest_sha=sha(PILOT/'pilot_manifest.json'),source_config_sha=sha(SOURCE/'preprocessing_config.json'),baseline_protocol_sha=sha(BASE/'protocol.json'),
 min_precision=MIN_PRECISION,target_recall=TARGET_RECALL,grid=RULE_GRID,seeds=MODEL_SEEDS,
 samples=SAMPLES,max_cpu_session_hours=CPU_SESSION_HOURS,threads=N_JOBS,batch=BATCH,
 validation_group_split='StratifiedGroupKFold 2, shuffle, seed42; canonical raw float32 hashes',
 objective='min recall of rare, then mean rare recall, then macro-F1; each rare precision >=0.30',
 evaluation_history='baseline validation/test already viewed',feature_selection_conditional=ENABLE_SHAP_IF_NEEDED)
PROTOCOL=json.loads(json.dumps(PROTOCOL));PH=oh(PROTOCOL);ROOT=Path('/kaggle/working/recall_experiments');ROOT.mkdir(exist_ok=True)
if RESUME_FROM:
 prior=Path(RESUME_FROM);assert read_json(prior/'protocol.json')==PROTOCOL,'Resume protocol/version mismatch'
 OUT=ROOT/prior.name
 if prior.resolve()!=OUT.resolve():
  assert not OUT.exists();shutil.copytree(prior,OUT)
else:
 OUT=ROOT/f'recall_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:8]}';OUT.mkdir()
 write_json(OUT/'protocol.json',PROTOCOL)
 write_json(OUT/'run_status.json',dict(status='in_progress',started_at=now(),protocol_hash=PH))
RUN_ID=OUT.name
SEARCH_LOCKED=(OUT/'frozen_candidates.json').exists()
(OUT/'input_snapshot').mkdir(exist_ok=True)
for f in ['preprocessing_config.json','feature_schema.json','label_mapping.json','source_manifest.json']:
 shutil.copy2(SOURCE/f,OUT/'input_snapshot'/f)
for f in ['protocol.json','environment.json','val_input_audit.json','test_input_audit.json','frozen_selection.json','val_float32_purge_audit.json','test_float32_purge_audit.json']:
 shutil.copy2(BASE/f,OUT/'input_snapshot'/('baseline_'+f))
ENV=dict(versions=VERSIONS,cpu=os.cpu_count(),threads=N_JOBS,
 ram_bytes=psutil.virtual_memory().total,platform=platform.platform(),device='cpu',paths=dict(source=str(SOURCE),pilot=str(PILOT),baseline=str(BASE)))
if not (OUT/'environment.json').exists():write_json(OUT/'environment.json',ENV)
sessions=read_json(OUT/'execution_sessions.json') if (OUT/'execution_sessions.json').exists() else []
sessions.append(dict(started_at=now(),environment=ENV));write_json(OUT/'execution_sessions.json',sessions)
(OUT/'requirements_runtime.txt').write_text('\n'.join(f'{k}=={v}' for k,v in VERSIONS.items() if k!='python')+'\n')
if OPTIONAL_VERSIONS['shap'] is not None:
 (OUT/'requirements_optional_shap.txt').write_text('shap=='+OPTIONAL_VERSIONS['shap']+'\n')
print('Output:',OUT)

class BudgetStop(Exception): pass
class Charge:
 def __init__(self,label,allow_final=False): self.label=label;self.allow_final=allow_final
 def __enter__(self):
  self.path=OUT/'budget_ledger.json';self.records=read_json(self.path) if self.path.exists() else []
  self.spent=sum(r['wall_seconds'] for r in self.records)
  if not self.allow_final and self.spent>=CPU_SESSION_HOURS*3600: raise BudgetStop('Hết ngân sách tìm kiếm; khóa từ các run đã xong.')
  self.start=time.perf_counter();self.work_id=uuid.uuid4().hex;self.stop=threading.Event()
  self.active=OUT/'active_work.json';assert not self.active.exists(),'Nested work accounting is not allowed'
  def heartbeat():
   write_json(self.active,dict(work=self.label,work_id=self.work_id,elapsed_seconds=time.perf_counter()-self.start,final_reporting=self.allow_final))
  heartbeat()
  def monitor_charge():
   while not self.stop.wait(15):heartbeat()
  self.thread=threading.Thread(target=monitor_charge,daemon=True);self.thread.start();return self
 def __exit__(self,*args):
  self.stop.set();self.thread.join();seconds=time.perf_counter()-self.start
  # Append to the latest ledger; supports nested accounting only at separate work boundaries.
  records=read_json(self.path) if self.path.exists() else []
  records.append(dict(work=self.label,work_id=self.work_id,wall_seconds=seconds,finished_at=now(),final_reporting=self.allow_final,failed=bool(args[0])))
  write_json(self.path,records);self.active.unlink(missing_ok=True)
def budget_left():
 records=read_json(OUT/'budget_ledger.json') if (OUT/'budget_ledger.json').exists() else []
 return max(0,CPU_SESSION_HOURS*3600-sum(r['wall_seconds'] for r in records))
# A heartbeat preserves charged work if Kaggle interrupts the kernel mid-trial.
active_work=OUT/'active_work.json'
if active_work.exists():
 abandoned=read_json(active_work);ledger=read_json(OUT/'budget_ledger.json') if (OUT/'budget_ledger.json').exists() else []
 if not any(r.get('work_id')==abandoned['work_id'] for r in ledger):
  ledger.append(dict(work=abandoned['work'],work_id=abandoned['work_id'],wall_seconds=abandoned['elapsed_seconds'],
   finished_at=now(),interrupted=True,final_reporting=abandoned['final_reporting'],precision='last 15-second heartbeat'))
  write_json(OUT/'budget_ledger.json',ledger)
 active_work.unlink()
BUDGET_STOPPED=False


## 2. Tập 1 triệu, validation cố định và chia nhóm 50/50
Hash chính vector 44 feature raw float32, chuẩn hóa dấu zero. Mẫu có cùng hash ở cùng phía. Index có ba hệ tọa độ: original processed split, baseline filtered split, và tập tune/confirm. Không dùng nhãn test để tạo hai phía.

Tập xác nhận không được dùng trong fitting, threshold search, chọn feature hoặc lựa chọn seed. Baseline đã từng báo metric toàn validation nên vẫn phải công bố giới hạn lịch sử này.


In [ ]:
class RowView:
 def __init__(self,array,rows):
  self.array=array;self.rows=np.asarray(rows,dtype=np.int64);self.shape=(len(self.rows),array.shape[1]);self.ndim=2
 def __len__(self): return len(self.rows)
 def __getitem__(self,key): return self.array[self.rows[key]]
def hashes(X,cols=None):
 h=np.empty(len(X),dtype='V16')
 for i in range(0,len(X),BATCH):
  a=np.array(X[i:i+BATCH],dtype='<f4',copy=True,order='C')
  if cols is not None: a=np.ascontiguousarray(a[:,cols])
  assert np.isfinite(a).all();a[a==0]=0
  h[i:i+len(a)]=np.frombuffer(b''.join(hashlib.blake2b(r.tobytes(),digest_size=16).digest() for r in a),dtype='V16')
 return h
def counts(y): return {c:int(n) for c,n in zip(CLASS_NAMES,np.bincount(y,minlength=8))}
info=next(x for x in PC['pilots'] if x['budget']==1_000_000);TRAIN_DIR=PILOT/info['directory']
TM=read_json(TRAIN_DIR/'pilot_manifest.json')
if VERIFY_HASHES:
 for f,h in {TM['features']['raw']['file']:TM['features']['raw']['sha256'],**TM['checksums_sha256']}.items(): assert sha(TRAIN_DIR/f)==h,f
XTRAIN=np.load(TRAIN_DIR/TM['features']['raw']['file'],mmap_mode='r',allow_pickle=False)
YTRAIN=np.load(TRAIN_DIR/'y_train.npy',allow_pickle=False)
TRAIN_META=pd.read_pickle(TRAIN_DIR/'sample_metadata_train.pkl').reset_index(drop=True)
assert XTRAIN.shape==(1_000_000,44) and np.array_equal(TRAIN_META.class_id.to_numpy(),YTRAIN)
assert counts(YTRAIN)==TM['class_support']
def verify_source_split(split):
 audit=read_json(BASE/(split+'_input_audit.json'));files=SC['artifacts'][split]
 if VERIFY_HASHES:
  for key,field in [('raw_file','raw_sha256'),('target_file','target_sha256'),('metadata_file','metadata_sha256')]:
   assert sha(SOURCE/files[key])==audit[field],f'{split}: {key} changed from baseline'
 return files
source_val=verify_source_split('val')
VRAW=np.load(SOURCE/source_val['raw_file'],mmap_mode='r',allow_pickle=False)
VY=np.load(SOURCE/source_val['target_file'],allow_pickle=False)
VMETA=pd.read_pickle(SOURCE/source_val['metadata_file']).reset_index(drop=True)
VKEEP=np.load(BASE/'val_float32_kept_processed_indices.npy',allow_pickle=False)
assert sha(BASE/'val_float32_kept_processed_indices.npy')==read_json(BASE/'val_float32_purge_audit.json')['kept_indices_sha256']
V=RowView(VRAW,VKEEP);YV=VY[VKEEP]
assert counts(YV)==read_json(BASE/'val_float32_purge_audit.json')['after_class_support']
assert np.array_equal(VMETA.class_id.to_numpy(),VY)
HVAL=hashes(V);HTRAIN=np.unique(hashes(XTRAIN))
assert np.intersect1d(HTRAIN,np.unique(HVAL)).size==0,'Train/validation overlap in actual float32 representation'
_,groups=np.unique(HVAL,return_inverse=True)
splitfile=OUT/'validation_partition.npz'
if splitfile.exists():
 z=np.load(splitfile,allow_pickle=False);TUNE=z['tune'];CONFIRM=z['confirm']
 assert np.array_equal(z['baseline_processed_rows'],VKEEP),'Validation mask changed since checkpoint'
 z.close()
else:
 fold=StratifiedGroupKFold(n_splits=2,shuffle=True,random_state=SEED)
 TUNE,CONFIRM=next(fold.split(np.zeros((len(YV),1),dtype=np.uint8),YV,groups))
 np.savez_compressed(splitfile,tune=TUNE,confirm=CONFIRM,baseline_processed_rows=VKEEP)
assert np.array_equal(np.sort(np.concatenate([TUNE,CONFIRM])),np.arange(len(YV)))
assert len(TUNE)+len(CONFIRM)==len(YV) and len(np.intersect1d(TUNE,CONFIRM))==0
assert np.intersect1d(groups[TUNE],groups[CONFIRM]).size==0
assert all(n>0 for n in counts(YV[TUNE]).values()) and all(n>0 for n in counts(YV[CONFIRM]).values())
XTUNE=RowView(VRAW,VKEEP[TUNE]);YTUNE=YV[TUNE]
write_json(OUT/'validation_partition.json',dict(protocol_hash=PH,partition_sha=sha(splitfile),
 original_val_rows=len(VY),baseline_filtered_rows=len(YV),tune_rows=len(TUNE),confirm_rows=len(CONFIRM),
 tune_support=counts(YTUNE),confirm_support=counts(YV[CONFIRM]),group_intersection=0,
 history='entire validation previously used in baseline; confirmation reserved for this new search only'))
display(pd.DataFrame(dict(class_name=CLASS_NAMES,tune=list(counts(YTUNE).values()),confirm=list(counts(YV[CONFIRM]).values()))))


## 3. Metrics, quy tắc dự đoán và checkpoint
`score_c = probability_c × multiplier_c`, argmax trả một trong 8 lớp. Điểm đã nhân không phải xác suất đã hiệu chỉnh. Precision không xác định khi không dự đoán lớp được tính bằng 0. Chọn ứng viên đủ precision ≥30% cả hai lớp, rồi tối đa recall thấp hơn; cấu hình không đủ precision chỉ xuất báo cáo, không được tuyên bố đạt mục tiêu.


In [ ]:
def metrics_from_labels(y,pred):
 cm=confusion_matrix(y,pred,labels=IDS);tp=np.diag(cm);support=cm.sum(axis=1);predcount=cm.sum(axis=0)
 precision=np.divide(tp,predcount,out=np.zeros(8,dtype=float),where=predcount>0)
 recall=np.divide(tp,support,out=np.zeros(8,dtype=float),where=support>0)
 f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(8,dtype=float),where=precision+recall>0)
 r=dict(macro_f1=float(f1.mean()),weighted_f1=float(np.dot(f1,support)/support.sum()),accuracy=float(tp.sum()/support.sum()),
  min_rare_precision=float(precision[RARE].min()),min_rare_recall=float(recall[RARE].min()),
  mean_rare_recall=float(recall[RARE].mean()),normal_false_alarm_rate=float(1-recall[0]),
  precision_BruteForce=float(precision[1]),precision_Web_Based=float(precision[7]),
  recall_BruteForce=float(recall[1]),recall_Web_Based=float(recall[7]),
  feasible=bool(np.all(precision[RARE]>=MIN_PRECISION)),
  target_met=bool(np.all(precision[RARE]>=MIN_PRECISION) and np.all(recall[RARE]>=TARGET_RECALL)))
 return r,cm,pd.DataFrame(dict(class_id=IDS,class_name=CLASS_NAMES,precision=precision,recall=recall,f1=f1,support=support,predicted_count=predcount))
def decide(prob,rule):
 prob=np.asarray(prob,dtype=np.float32)
 w=np.ones(8,dtype=np.float32);w[1]=rule[0];w[7]=rule[1]
 return np.argmax(prob*w,axis=1).astype(np.uint8)
def rank(r):
 if r['feasible']: return (1,r['min_rare_recall'],r['mean_rare_recall'],r['macro_f1'])
 return (0,r['min_rare_precision'],r['min_rare_recall'],r['macro_f1'])
def rule_search(prob,y):
 records=[]
 for bf in RULE_GRID:
  for web in RULE_GRID:
   m,_,_=metrics_from_labels(y,decide(prob,[bf,web]));records.append(dict(multiplier_BruteForce=bf,multiplier_Web_Based=web,**m))
 # Stable ascending multiplier order resolves exact ties without more aggressive decisions.
 best=max(records,key=rank)
 return [best['multiplier_BruteForce'],best['multiplier_Web_Based']],best,pd.DataFrame(records)
def predict_batches(model,X,cols):
 assert np.array_equal(model.classes_,IDS),'Estimator class order mismatch'
 prob=np.empty((len(X),8),dtype=np.float32);seconds=0.0
 for i in range(0,len(X),BATCH):
  a=np.ascontiguousarray(X[i:i+BATCH][:,cols],dtype=np.float32)
  t=time.perf_counter();p=model.predict_proba(a);seconds+=time.perf_counter()-t
  assert p.shape==(len(a),8) and np.isfinite(p).all() and np.allclose(p.sum(axis=1),1,atol=1e-5)
  prob[i:i+len(a)]=p
 return prob,seconds
RESULTS={};CANDIDATES=[]
def remember(result):
 RESULTS[result['id']]=result
 pd.DataFrame([dict(id=r['id'],stage=r['stage'],model=r['spec']['model'],sample=r['spec']['sample'],
  weight=r['spec']['weight'],seed=r['spec']['seed'],fit_seconds=r['fit_seconds'],**r['tune_metrics']) for r in RESULTS.values()]).to_csv(OUT/'search_results.csv',index=False)
 return result
def cache_result(folder):
 f=folder/'result.json'
 if not f.exists(): return None
 r=read_json(f);assert r['protocol_hash']==PH
 for name,digest in r['checksums'].items(): assert sha(folder/name)==digest, f'Cached file changed: {folder/name}'
 return remember(r)
def save_result(folder,result):
 # All trials include complete 8-class tables, before and after the decision rule.
 prob=np.load(folder/'tune_probabilities.npy',mmap_mode='r',allow_pickle=False)
 for tag,pred in [('unadjusted',np.argmax(prob,axis=1)),('adjusted',decide(prob,result['rule']))]:
  _,cm,table=metrics_from_labels(YTUNE,pred);table.to_csv(folder/('tune_per_class_'+tag+'.csv'),index=False)
  pd.DataFrame(cm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(folder/('tune_confusion_'+tag+'.csv'),index_label='true_class')
 result['checksums']={p.name:sha(p) for p in sorted(folder.iterdir()) if p.is_file() and p.name not in ['result.json','status.json']}
 write_json(folder/'result.json',result);return remember(result)
def lookup_folder(result): return OUT/'trials'/result['id']
def spec_for(model,sample='original',weight='none',params=None,features=None,seed=42):
 p=dict(BASE_PARAMS[model] if params is None else params);p.pop('random_state',None)
 if model in ['RF','XGBoost']:p['n_jobs']=N_JOBS
 return dict(model=model,sample=sample,weight=weight,params=p,features=list(range(44)) if features is None else list(map(int,features)),seed=seed)

def sampling_indices(name):
 folder=OUT/'sampling';folder.mkdir(exist_ok=True)
 p=folder/(name+'.npy')
 if p.exists() and (folder/(name+'.json')).exists():
  assert sha(p)==read_json(folder/(name+'.json'))['indices_sha'],'Sampling checkpoint changed'
  return np.load(p,allow_pickle=False)
 if SAMPLES[name] is None: indices=np.arange(len(YTRAIN),dtype=np.int64)
 else:
  parts=[];rng=np.random.default_rng(SEED)
  raw=TRAIN_META.original_label.astype(str).to_numpy()
  for c in IDS:
   pool=np.flatnonzero(YTRAIN==c);cap=min(len(pool),SAMPLES[name].get(int(c),len(pool)))
   if cap==len(pool): parts.append(pool);continue
   labels,ct=np.unique(raw[pool],return_counts=True);q=ct*cap/ct.sum();n=np.floor(q).astype(int)
   order=np.argsort(-(q-n),kind='stable');n[order[:cap-int(n.sum())]]+=1
   for label,k in zip(labels,n): parts.append(rng.choice(pool[raw[pool]==label],size=k,replace=False))
  indices=np.sort(np.concatenate(parts)).astype(np.int64)
 assert len(np.unique(indices))==len(indices)
 for c in RARE: assert int((YTRAIN[indices]==c).sum())==int((YTRAIN==c).sum())
 np.save(p,indices,allow_pickle=False)
 table=TRAIN_META.iloc[indices].groupby(['class_id','original_label'],observed=True).size().reset_index(name='support')
 table.to_csv(folder/(name+'_support_by_original_label.csv'),index=False)
 write_json(folder/(name+'.json'),dict(sample=name,rows=len(indices),support=counts(YTRAIN[indices]),seed=SEED,indices_sha=sha(p),stratified_by='original label within DDoS/DoS'))
 return indices

def train_run(spec,stage,rule_override=None):
 key=stage+'_'+oh(dict(spec=spec,rule_override=rule_override))[:16];folder=OUT/'trials'/key
 cached=cache_result(folder)
 if cached:return cached
 if (OUT/'frozen_candidates.json').exists(): raise RuntimeError('Candidates frozen: no refit or search allowed.')
 if budget_left()<300:raise BudgetStop('Dành thời gian còn lại cho khóa và báo cáo; không mở fit mới.')
 folder.mkdir(parents=True,exist_ok=True);write_json(folder/'status.json',dict(status='in_progress',started_at=now()))
 print('Fit',key,spec['model'],spec['sample'],spec['weight'],'seed',spec['seed'],flush=True)
 with Charge(key):
  idx=sampling_indices(spec['sample']);cols=spec['features']
  X=np.ascontiguousarray(XTRAIN[idx][:,cols],dtype=np.float32);y=YTRAIN[idx]
  alpha={'none':0.0,'sqrt':0.5,'balanced':1.0}[spec['weight']]
  ct=np.bincount(y,minlength=8);cw=(len(y)/(8*ct))**alpha;cw/=np.dot(cw,ct)/len(y)
  weights=None if alpha==0 else cw[y]
  params={**spec['params'],'random_state':spec['seed']}
  model={'DT':DecisionTreeClassifier,'RF':RandomForestClassifier,'XGBoost':XGBClassifier}[spec['model']](**params)
  process=psutil.Process();base_rss=process.memory_info().rss;peak=[base_rss];event=threading.Event()
  def monitor():
   while not event.wait(.1):peak[0]=max(peak[0],process.memory_info().rss)
  thread=threading.Thread(target=monitor,daemon=True);thread.start()
  try:
   with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always');t=time.perf_counter();model.fit(X,y,sample_weight=weights);fit_seconds=time.perf_counter()-t
  finally:
   peak[0]=max(peak[0],process.memory_info().rss);event.set();thread.join()
  assert np.array_equal(model.classes_,IDS)
  p,seconds=predict_batches(model,XTUNE,cols)
  if rule_override is None: rule,m,grid=rule_search(p,YTUNE);grid.to_csv(folder/'rule_grid.csv',index=False)
  else:rule=list(rule_override);m,_,_=metrics_from_labels(YTUNE,decide(p,rule))
  before=p[:20].copy();joblib.dump(model,folder/'model.joblib',compress=3)
  if spec['model']=='XGBoost':model.save_model(folder/'model.ubj')
  np.save(folder/'tune_probabilities.npy',p,allow_pickle=False)
  m0,_,_=metrics_from_labels(YTUNE,np.argmax(p,axis=1))
  # Reproducible stratified training audit within each candidate sample; never confirm/test.
  rng=np.random.default_rng(SEED);sample=np.concatenate([rng.choice(np.flatnonzero(y==c),min(2000,int((y==c).sum())),replace=False) for c in IDS])
  ptr=model.predict_proba(X[sample]);mt,_,_=metrics_from_labels(y[sample],decide(ptr,rule))
  write_json(folder/'train_audit.json',dict(support=counts(y[sample]),sample_policy='up to 2000/class; audit only; not population performance',metrics=mt))
  del X,weights,model;gc.collect();model=joblib.load(folder/'model.joblib')
  check,_=predict_batches(model,RowView(XTUNE.array,XTUNE.rows[:20]),cols)
  assert np.allclose(check,before,atol=1e-7,rtol=1e-6);del model;gc.collect()
  result=dict(id=key,stage=stage,spec=spec,protocol_hash=PH,model_path=str((folder/'model.joblib').relative_to(OUT)),
   borrowed_baseline=False,fit_environment=ENV,fit_seconds=fit_seconds,fit_peak_rss_mib=peak[0]/2**20,fit_baseline_rss_mib=base_rss/2**20,
   tune_predict_seconds=seconds,rule=rule,tune_metrics=m,unadjusted_tune_metrics=m0,
   class_weights=dict(zip(CLASS_NAMES,map(float,cw))),train_rows=len(idx),warning_messages=[str(w.message) for w in caught])
  write_json(folder/'status.json',dict(status='complete',finished_at=now()));return save_result(folder,result)


## 4. A — dùng lại 9 model đã có để tìm hệ số quyết định
Chỉ lấy xác suất ở index tune từ baseline; không đọc xác suất confirm/test trong tìm kiếm. Model baseline phải đúng cùng preprocessing, feature order, class order và version. Báo cáo lỗi theo nhãn Web-Based gốc; đây là phân tích metadata, không huấn luyện 34 lớp.


In [ ]:
def baseline_trial(name,weight):
 old=f'{name}_n1000000_{weight}_s42';source=BASE/'experiments'/old
 assert read_json(source/'result.json')['status']=='complete'
 key='A_'+old;folder=OUT/'trials'/key;cached=cache_result(folder)
 if cached:return cached
 assert not (OUT/'frozen_candidates.json').exists()
 folder.mkdir(parents=True,exist_ok=True)
 with Charge(key):
  r=read_json(source/'result.json');cfg=read_json(source/'config.json')
  assert cfg['features']==FEATURES and cfg['class_order']==CLASS_NAMES and cfg['budget']==1_000_000
  assert cfg['model_params']==BASE_PARAMS[name] and cfg['weight_mode']==weight
  if VERIFY_HASHES:
   for rel in ['config.json','model.joblib','validation/probabilities.npy']:
    assert sha(source/rel)==r['artifact_hashes'][rel]
  p=np.load(source/'validation/probabilities.npy',mmap_mode='r',allow_pickle=False)
  assert p.shape==(len(YV),8)
  tune=np.array(p[TUNE],dtype=np.float32);rule,m,grid=rule_search(tune,YTUNE)
  grid.to_csv(folder/'rule_grid.csv',index=False);np.save(folder/'tune_probabilities.npy',tune,allow_pickle=False)
  shutil.copy2(source/'model.joblib',folder/'model.joblib')
  if (source/'model.ubj').exists():shutil.copy2(source/'model.ubj',folder/'model.ubj')
  m0,_,_=metrics_from_labels(YTUNE,np.argmax(tune,axis=1))
  spec=spec_for(name,weight=weight)
  spec['params']=dict(cfg['model_params']);spec['params'].pop('random_state',None)
  result=dict(id=key,stage='A',spec=spec,protocol_hash=PH,model_path=str((folder/'model.joblib').relative_to(OUT)),borrowed_baseline=True,
   fit_seconds=r['fit_seconds'],fit_peak_rss_mib=r['fit_peak_rss_mib'],tune_predict_seconds=None,
   rule=rule,tune_metrics=m,unadjusted_tune_metrics=m0,class_weights=cfg['class_weights'],train_rows=1_000_000,baseline_experiment_id=old,fit_environment_reference='input_snapshot/baseline_environment.json',fit_timing_scope='borrowed from baseline; not repeated in this session')
  predicted=decide(tune,rule)
  original=VMETA.iloc[VKEEP[TUNE]].original_label.astype(str).to_numpy()
  error_table=pd.DataFrame(dict(original_label=original,true_class=YTUNE,predicted_class=predicted))
  error_table.groupby(['original_label','true_class','predicted_class']).size().reset_index(name='count').to_csv(folder/'errors_by_original_label.csv',index=False)
  return save_result(folder,result)
if SEARCH_LOCKED:
 for checkpoint in sorted((OUT/'trials').glob('*/result.json')):cache_result(checkpoint.parent)
else:
 try:
  for name in MODELS:
   for weight in ['none','sqrt','balanced']:baseline_trial(name,weight)
 except BudgetStop as e:BUDGET_STOPPED=True;print(e)
print('Stage A saved:',len([r for r in RESULTS.values() if r['stage']=='A']))
display(pd.read_csv(OUT/'search_results.csv').sort_values(['feasible','min_rare_recall'],ascending=False))


## 5. B — giảm DDoS/DoS và thử trọng số
18 fit mới cho moderate/strong × 3 thuật toán × 3 trọng số. Chín original được dùng lại. Không loại lớp hiếm hoặc gộp thành binary. Mọi model dùng đúng validation mask/split đã lưu.


In [ ]:
if not BUDGET_STOPPED and not SEARCH_LOCKED:
 try:
  for sample in ['moderate','strong']:
   for name in MODELS:
    for weight in ['none','sqrt','balanced']:train_run(spec_for(name,sample,weight),'B')
 except BudgetStop as e:BUDGET_STOPPED=True;print(e)
BEST_BY_MODEL={}
for name in MODELS:
 choices=[r for r in RESULTS.values() if r['spec']['model']==name and r['spec']['seed']==42]
 if choices:BEST_BY_MODEL[name]=max(choices,key=lambda r:rank(r['tune_metrics']))
if not SEARCH_LOCKED:write_json(OUT/'stage_B_selected.json',{k:dict(id=r['id'],spec=r['spec'],rule=r['rule'],metrics=r['tune_metrics']) for k,r in BEST_BY_MODEL.items()})


## 6. C — tuning có giới hạn
DT 6 cấu hình; RF/XGBoost mỗi loại 12 cấu hình lấy từ ParameterSampler seed42. Tuning dùng phương án sample/weight tốt nhất tại B. Số cây cố định trong từng trial, không dùng confirm để early stop. Không đổi số lớp.


In [ ]:
spaces={
 'DT':dict(max_depth=[12,20,None],min_samples_leaf=[1,2]),
 'RF':dict(n_estimators=[200,400],max_depth=[20,30,None],min_samples_leaf=[1,2,4],max_features=['sqrt',0.5,1.0]),
 'XGBoost':dict(max_depth=[4,6,8],min_child_weight=[1,3,5],learning_rate=[0.05,0.1],n_estimators=[300,600],max_bin=[256,512])}
if not BUDGET_STOPPED and not SEARCH_LOCKED:
 try:
  for name in MODELS:
   if name not in BEST_BY_MODEL:continue
   base=BEST_BY_MODEL[name]['spec']
   for params in ParameterSampler(spaces[name],n_iter=6 if name=='DT' else 12,random_state=SEED):
    merged={**base['params'],**params}
    train_run(spec_for(name,base['sample'],base['weight'],merged),'C')
 except BudgetStop as e:BUDGET_STOPPED=True;print(e)
BEST_BY_MODEL={name:max([r for r in RESULTS.values() if r['spec']['model']==name and r['spec']['seed']==42],key=lambda r:rank(r['tune_metrics'])) for name in BEST_BY_MODEL}


## 7. D — SHAP từ train, chỉ khi chưa đạt mục tiêu
Hai thuật toán tốt nhất. XGBoost dùng Tree SHAP native `pred_contribs` trong raw margin. RF/DT dùng TreeExplainer nếu SHAP có sẵn; nếu thiếu hoặc không tương thích, ghi skipped và dùng tập feature đầy đủ, không giả tên phương pháp khác là SHAP.

Mẫu train tối đa 200 mỗi lớp, lưu ID. Mức quan trọng từng lớp dùng các mẫu train thuộc lớp đó; macro-average qua 8 lớp. Top20/top30 cộng top5 của từng lớp hiếm. Trước fit, audit giao nhau train/validation và tune/confirmation trong biểu diễn float32 rút gọn; nếu có thì loại bộ feature đó để giữ nguyên evaluation, không purge riêng theo model. Lưu overlap count, không cố làm sạch cho điểm đẹp.


In [ ]:
def shap_sets(result):
 folder=OUT/'feature_selection'/result['id'];folder.mkdir(parents=True,exist_ok=True)
 file=folder/'selection.json'
 if file.exists():return read_json(file)['feature_sets']
 assert not (OUT/'frozen_candidates.json').exists()
 with Charge('SHAP_'+result['id']):
  idx=sampling_indices(result['spec']['sample']);y=YTRAIN[idx];rng=np.random.default_rng(SEED)
  sample=np.concatenate([rng.choice(np.flatnonzero(y==c),min(200,int((y==c).sum())),replace=False) for c in IDS])
  source_rows=idx[sample];X=np.ascontiguousarray(XTRAIN[source_rows],dtype=np.float32)
  model=joblib.load(lookup_folder(result)/'model.joblib')
  if result['spec']['model']=='XGBoost':
   contrib=model.get_booster().predict(xgboost.DMatrix(X),pred_contribs=True,strict_shape=True)
   assert contrib.shape==(len(X),8,45);values=np.transpose(contrib[:,:,:-1],(0,2,1));method='XGBoost native Tree SHAP raw margin'
  else:
   import shap
   explainer=shap.TreeExplainer(model,feature_perturbation='tree_path_dependent',model_output='raw')
   values=explainer.shap_values(X,check_additivity=True)
   if isinstance(values,list):values=np.stack(values,axis=-1)
   values=np.asarray(values);assert values.shape==(len(X),44,8)
   method='shap.TreeExplainer raw tree output'
  importance=np.vstack([np.abs(values[y[sample]==c,:,c]).mean(axis=0) for c in IDS])
  overall=importance.mean(axis=0);order=np.argsort(-overall,kind='stable')
  protect=set(np.argsort(-importance[1],kind='stable')[:5])|set(np.argsort(-importance[7],kind='stable')[:5])
  sets=[sorted(map(int,set(order[:k])|protect)) for k in [20,30]]
  np.save(folder/'processed_train_indices.npy',source_rows,allow_pickle=False)
  table=pd.DataFrame(importance.T,index=FEATURES,columns=CLASS_NAMES);table['macro_mean']=overall
  table.to_csv(folder/'importance_by_class.csv',index_label='feature')
  write_json(file,dict(method=method,feature_sets=sets,protected_indices=sorted(map(int,protect)),
   sample_support=counts(y[sample]),sample_seed=SEED,source_model=result['id'],optional_versions=OPTIONAL_VERSIONS))
  del model,values;gc.collect();return sets

if ENABLE_SHAP_IF_NEEDED and not BUDGET_STOPPED and not SEARCH_LOCKED and not any(r['tune_metrics']['target_met'] for r in BEST_BY_MODEL.values()):
 top=sorted(BEST_BY_MODEL.values(),key=lambda r:rank(r['tune_metrics']),reverse=True)[:2]
 for r in top:
  try:
   for cols in shap_sets(r):
    with Charge('audit_reduced_features_'+r['id']):
     # Compare to all validation rows; no predictions/labels/metrics from confirm are used.
     # Sample/feature variants are subsets of the largest train pilot.
     th=np.unique(hashes(XTRAIN,cols));projected_val=hashes(V,cols);vh=np.unique(projected_val)
     overlap=int(np.intersect1d(th,vh).size)
     confirmation_overlap=int(np.intersect1d(np.unique(projected_val[TUNE]),np.unique(projected_val[CONFIRM])).size)
    audit=OUT/'feature_selection'/r['id']/('overlap_'+oh(cols)[:8]+'.json')
    write_json(audit,dict(features=cols,count=len(cols),train_val_shared_vectors=overlap,tune_confirm_shared_vectors=confirmation_overlap,action='reject' if overlap or confirmation_overlap else 'allow'))
    if overlap or confirmation_overlap:
     print('Reject reduced feature set:',len(cols),'train/val:',overlap,'tune/confirm:',confirmation_overlap);continue
    base=r['spec'];train_run(spec_for(base['model'],base['sample'],base['weight'],base['params'],cols),'D')
  except BudgetStop as e:BUDGET_STOPPED=True;print(e);break
  except (ImportError,ValueError,AssertionError) as e:
   write_json(OUT/'feature_selection'/r['id']/'skipped.json',dict(reason=repr(e),action='retain full feature results'))
   print('SHAP stage skipped for',r['spec']['model'],repr(e))


## 8. Ba seed và khóa tối đa hai ứng viên
Chọn hai trial seed42 tốt nhất. Với mỗi trial, chạy seed2026/3407 trên cùng sample/features/params, **giữ hệ số quyết định của seed42** để đo ổn định thay vì chỉnh thêm cho từng seed. Xếp ứng viên bằng recall trung bình qua các seed; precision phải đạt 30% ở từng seed đã chạy. Seed42 luôn là model bàn giao; không chọn seed có điểm đẹp nhất.

Nếu hết ngân sách, công bố số seed thực chạy. Khi đã có `frozen_candidates.json`, resume chỉ dùng checkpoint, không mở thêm tìm kiếm.


In [ ]:
frozen_file=OUT/'frozen_candidates.json'
if frozen_file.exists():
 FROZEN=read_json(frozen_file);assert FROZEN['protocol_hash']==PH
else:
 eligible=[r for r in RESULTS.values() if r['spec']['seed']==42 and r['tune_metrics']['feasible']]
 assert eligible,'Không có model đủ precision30%; xem search_results.csv, không công bố đã đạt mục tiêu.'
 top=sorted(eligible,key=lambda r:rank(r['tune_metrics']),reverse=True)[:2];groups=[]
 for r in top:
  seed_results=[r]
  for seed in MODEL_SEEDS[1:]:
   if BUDGET_STOPPED:break
   try:
    spec={**r['spec'],'seed':seed};seed_results.append(train_run(spec,'seeds',r['rule']))
   except BudgetStop as e:BUDGET_STOPPED=True;print(e);break
  score={k:float(np.mean([x['tune_metrics'][k] for x in seed_results])) for k in ['recall_BruteForce','recall_Web_Based','macro_f1']}
  score['min_rare_recall']=min(score['recall_BruteForce'],score['recall_Web_Based'])
  score['mean_rare_recall']=(score['recall_BruteForce']+score['recall_Web_Based'])/2
  score['min_rare_precision']=min(x['tune_metrics']['min_rare_precision'] for x in seed_results)
  score['feasible']=score['min_rare_precision']>=MIN_PRECISION
  groups.append(dict(trial_id=r['id'],model_path=r['model_path'],spec=r['spec'],rule=r['rule'],
   model_sha=sha(lookup_folder(r)/'model.joblib'),seed_trial_ids=[x['id'] for x in seed_results],
   seeds=[x['spec']['seed'] for x in seed_results],selection_metrics=score,
   recall_std_BruteForce=float(np.std([x['tune_metrics']['recall_BruteForce'] for x in seed_results])),
   recall_std_Web_Based=float(np.std([x['tune_metrics']['recall_Web_Based'] for x in seed_results]))))
 groups=sorted(groups,key=lambda r:rank(r['selection_metrics']),reverse=True)
 write_json(OUT/'prefreeze_seed_results.json',groups)
 if not groups[0]['selection_metrics']['feasible']:
  write_json(OUT/'run_status.json',dict(run_id=RUN_ID,status='search_failed_precision_stability',protocol_hash=PH,
   reason='Neither finalist meets rare precision floor across actual seeds; confirmation/test not opened'))
  raise RuntimeError('Seed stability failed precision floor. Reports saved in prefreeze_seed_results.json; do not open confirmation.')
 FROZEN=dict(protocol_hash=PH,frozen_at=now(),candidates=groups,primary_trial_id=groups[0]['trial_id'],
   choice_source='tune only, seed42 model fixed, mean recall across actual seeds',confirmation_used=False,test_used=False)
 write_json(frozen_file,FROZEN)
print('Primary frozen:',FROZEN['primary_trial_id'])
display(pd.DataFrame([dict(id=r['trial_id'],seeds=str(r['seeds']),**r['selection_metrics']) for r in FROZEN['candidates']]))


## 9. Báo cáo từng split, CI recall và benchmark
Bảng trước/sau quy tắc, support, dự đoán có ID, xác suất nguyên gốc, confusion matrix, Wilson95% recall. CI chỉ mô tả bất định theo mẫu; dữ liệu cùng phiên có thể phụ thuộc, CI không chứng minh độc lập.

Benchmark warm predict_proba + decision, batch1/1024 ×30; không gồm CSV/XAI/UI. Phần xác nhận chỉ được mở sau freeze, không thay lựa chọn theo điểm này.


In [ ]:
def wilson(tp,n):
 if n==0:return [0.,0.]
 z=1.959963984540054;p=tp/n;d=1+z*z/n
 center=(p+z*z/(2*n))/d;delta=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
 return [float(center-delta),float(center+delta)]
def save_assessment(entry,X,y,processed_ids,source_ids,original_labels,split):
 folder=OUT/'assessments'/split/entry['trial_id'];file=folder/'result.json'
 if file.exists():
  r=read_json(file);assert r['frozen_sha']==sha(frozen_file)
  for rel,digest in r['checksums'].items():assert sha(folder/rel)==digest
  return r
 folder.mkdir(parents=True,exist_ok=True)
 with Charge('final_'+split+'_'+entry['trial_id'],allow_final=True):
  assert sha(OUT/entry['model_path'])==entry['model_sha']
  t=time.perf_counter();model=joblib.load(OUT/entry['model_path']);load_seconds=time.perf_counter()-t
  p,seconds=predict_batches(model,X,entry['spec']['features']);pred=decide(p,entry['rule'])
  m,cm,table=metrics_from_labels(y,pred);m0,cm0,table0=metrics_from_labels(y,np.argmax(p,axis=1))
  table0.to_csv(folder/'per_class_metrics_unadjusted.csv',index=False)
  pd.DataFrame(cm0,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(folder/'confusion_matrix_unadjusted.csv',index_label='true_class')
  table['recall_ci95_low']=[wilson(cm[c,c],int(cm[c].sum()))[0] for c in IDS]
  table['recall_ci95_high']=[wilson(cm[c,c],int(cm[c].sum()))[1] for c in IDS]
  table.to_csv(folder/'per_class_metrics.csv',index=False)
  pd.DataFrame(cm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(folder/'confusion_matrix_counts.csv',index_label='true_class')
  norm=cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
  pd.DataFrame(norm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(folder/'confusion_matrix_normalized.csv',index_label='true_class')
  fig,axes=plt.subplots(1,2,figsize=(17,7))
  for ax,values,title in zip(axes,[cm,norm],['Counts','Recall']):
   im=ax.imshow(values,cmap='Blues');fig.colorbar(im,ax=ax,fraction=.046)
   ax.set(xticks=range(8),yticks=range(8),xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,xlabel='Predicted',ylabel='True',title=split+' '+title)
   plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
   for i in IDS:
    for j in IDS:ax.text(j,i,str(int(values[i,j])) if title=='Counts' else f'{values[i,j]:.2f}',ha='center',va='center',fontsize=7,color='white' if values[i,j]>values.max()/2 else 'black')
  fig.tight_layout();fig.savefig(folder/'confusion_matrix.png',dpi=160);plt.close(fig)
  np.save(folder/'probabilities.npy',p,allow_pickle=False)
  pd.DataFrame(dict(evaluation_row_index=np.arange(len(y)),processed_row_index=processed_ids,source_row_index=source_ids,
   true_class_id=y,predicted_class_id=pred,source_split=split,original_label=original_labels)).to_csv(folder/'predictions.csv.gz',index=False,compression='gzip')
  benchmarks=[]
  for size in [1,min(1024,len(X))]:
   idx=np.random.default_rng(SEED+size).choice(len(X),size=size,replace=False)
   a=np.ascontiguousarray(X[idx][:,entry['spec']['features']],dtype=np.float32)
   for _ in range(3):decide(model.predict_proba(a),entry['rule'])
   times=[]
   for _ in range(30):
    t=time.perf_counter();decide(model.predict_proba(a),entry['rule']);times.append(time.perf_counter()-t)
   benchmarks.append(dict(batch=size,repeats=30,median_ms=float(np.median(times)*1000),p95_ms=float(np.percentile(times,95)*1000),scope='CPU warm estimator.predict_proba plus decision'))
  write_json(folder/'inference_benchmark.json',benchmarks)
  record=dict(split=split,trial_id=entry['trial_id'],frozen_sha=sha(frozen_file),metrics=m,unadjusted_metrics=m0,
   rows=len(y),support=counts(y),model_load_seconds=load_seconds,predict_proba_seconds=seconds,
   historical_evaluation_seen=True,measurement_environment=ENV,finished_at=now())
  record['checksums']={f.name:sha(f) for f in sorted(folder.iterdir()) if f.is_file() and f.name!='result.json'}
  write_json(file,record);del model;gc.collect();return record

ASSESSMENTS=[]
if RUN_CONFIRMATION:
 marker=OUT/'confirmation_opened.json'
 if not marker.exists():write_json(marker,dict(at=now(),frozen_sha=sha(frozen_file),purpose='reserved confirmation in this round; previously part of baseline validation'))
 assert read_json(marker)['frozen_sha']==sha(frozen_file)
 xc=RowView(VRAW,VKEEP[CONFIRM]);yc=YV[CONFIRM];meta=VMETA.iloc[VKEEP[CONFIRM]]
 for entry in FROZEN['candidates']:
  ASSESSMENTS.append(save_assessment(entry,xc,yc,VKEEP[CONFIRM],meta.source_row_index.to_numpy(),meta.original_label.astype(str).to_numpy(),'confirmation'))
 pd.DataFrame([dict(id=r['trial_id'],**r['metrics']) for r in ASSESSMENTS]).to_csv(OUT/'confirmation_summary.csv',index=False)
 display(pd.read_csv(OUT/'confirmation_summary.csv'))
else:print('Confirmation pending by configuration; not complete.')


## 10. Test cũ — đánh giá lại sau khóa, không chọn lại ứng viên
Đọc đúng mask test baseline, kiểm tra hash và giao nhau. Nếu bộ feature rút gọn trùng với test, bỏ báo cáo test cho ứng viên đó và ghi lý do; không sửa ứng viên theo test. Phân tích report theo lớp còn lại trong confirmation, không tuyên bố test độc lập mới.


In [ ]:
if RUN_OLD_TEST_REPORT:
 assert frozen_file.exists()
 marker=OUT/'old_test_opened.json'
 if not marker.exists():write_json(marker,dict(at=now(),frozen_sha=sha(frozen_file),history='test already viewed in baseline; no model selection from this report'))
 assert read_json(marker)['frozen_sha']==sha(frozen_file)
 artifact=verify_source_split('test');TRAW=np.load(SOURCE/artifact['raw_file'],mmap_mode='r',allow_pickle=False)
 TY=np.load(SOURCE/artifact['target_file'],allow_pickle=False);META=pd.read_pickle(SOURCE/artifact['metadata_file']).reset_index(drop=True)
 keep=np.load(BASE/'test_float32_kept_processed_indices.npy',allow_pickle=False)
 assert sha(BASE/'test_float32_kept_processed_indices.npy')==read_json(BASE/'test_float32_purge_audit.json')['kept_indices_sha256']
 xt=RowView(TRAW,keep);yt=TY[keep]
 assert counts(yt)==read_json(BASE/'test_float32_purge_audit.json')['after_class_support']
 assert np.array_equal(META.class_id.to_numpy(),TY)
 assert np.intersect1d(HTRAIN,np.unique(hashes(xt))).size==0
 assert np.intersect1d(np.unique(HVAL),np.unique(hashes(xt))).size==0
 reports=[]
 for entry in FROZEN['candidates']:
  cols=entry['spec']['features']
  if len(cols)<44:
   th=np.unique(hashes(XTRAIN,cols));vh=np.unique(hashes(V,cols));te=np.unique(hashes(xt,cols))
   overlap=int(np.intersect1d(th,te).size+np.intersect1d(vh,te).size)
   if overlap:
    write_json(OUT/'assessments'/'old_test'/entry['trial_id']/'skipped.json',dict(reason='reduced representation overlaps with old test',overlap_groups=overlap,action='no reselection or per-model purge'))
    continue
  reports.append(save_assessment(entry,xt,yt,keep,META.iloc[keep].source_row_index.to_numpy(),META.iloc[keep].original_label.astype(str).to_numpy(),'old_test'))
 summary=pd.DataFrame([dict(id=r['trial_id'],**r['metrics']) for r in reports])
 if summary.empty:summary=pd.DataFrame(columns=['id','macro_f1','recall_BruteForce','recall_Web_Based','target_met'])
 summary.to_csv(OUT/'old_test_summary.csv',index=False);display(summary)


## 11. Pipeline bàn giao: preprocessing + estimator + decision
Module `recall_inference.py` phải được import trước khi nạp pipeline. `pipeline.predict` dùng hệ số đã khóa; `predict_proba` trả xác suất model nguyên gốc, có thể không cùng argmax với nhãn sau điều chỉnh. Hải lấy estimator ở `pipeline.named_steps['model'].base_estimator`, đúng feature/order và raw float32. Giải thích estimator và quy tắc quyết định phải được trình bày riêng.

Wrapper có `fit` để tuân thủ giao diện estimator scikit-learn; phương thức này chỉ kiểm tra model đã fit, không huấn luyện lại model đã khóa. Nếu gặp lỗi cũ tại bước này, thay toàn bộ code cell 11 bằng bản đã sửa, chạy lại cell 11 rồi cell 12 trong phiên hiện tại.


In [ ]:
INFERENCE_MODULE='"""8-class recall pipeline. Import this module before joblib.load."""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.base import BaseEstimator, ClassifierMixin\nfrom sklearn.utils.validation import check_is_fitted\nfrom sklearn.exceptions import NotFittedError\n\nclass DecisionRuleClassifier(ClassifierMixin, BaseEstimator):\n    def __init__(self, base_estimator, multipliers=(1.,1.)):\n        self.base_estimator=base_estimator\n        self.multipliers=multipliers\n    @property\n    def classes_(self): return self.base_estimator.classes_\n    @property\n    def n_features_in_(self): return self.base_estimator.n_features_in_\n    def fit(self, X=None, y=None):\n        # This wrapper accepts an already fitted estimator; never refit the frozen model.\n        check_is_fitted(self.base_estimator)\n        return self\n    def __sklearn_is_fitted__(self):\n        try:\n            check_is_fitted(self.base_estimator)\n        except NotFittedError:\n            return False\n        return True\n    def predict_proba(self,X): return self.base_estimator.predict_proba(X)\n    def decision_function(self,X):\n        p=np.asarray(self.predict_proba(X),dtype=np.float32);w=np.ones(8,dtype=np.float32);w[1],w[7]=self.multipliers\n        return p*w\n    def predict(self,X): return self.classes_[self.decision_function(X).argmax(axis=1)]\n\ndef load_bundle(path):\n    root=Path(path)\n    return joblib.load(root/\'pipeline.joblib\'),json.loads((root/\'config.json\').read_text())\ndef predict_frame(pipeline,config,frame):\n    required=config[\'feature_names\']\n    if frame.columns.duplicated().any():raise ValueError(\'Duplicate columns\')\n    missing=sorted(set(required)-set(frame.columns))\n    if missing:raise ValueError(f\'Missing features: {missing}\')\n    X=frame.loc[:,required].apply(pd.to_numeric,errors=\'raise\').astype(np.float64)\n    X=X.replace([np.inf,-np.inf],np.nan)\n    p=pipeline.predict_proba(X);pred=pipeline.predict(X)\n    if not np.isfinite(p).all():raise ValueError(\'Nonfinite probabilities\')\n    result=pd.DataFrame({\'input_row\':np.arange(len(frame)),\'predicted_class_id\':pred,\n        \'predicted_class_name\':[config[\'class_names\'][int(i)] for i in pred]})\n    for i,c in enumerate(config[\'class_names\']): result[\'probability_\'+c]=p[:,i]\n    return result\n\ndef predict_csv(bundle,csv_path,output,chunksize=25000):\n    pipeline,config=load_bundle(bundle);offset=0\n    for i,frame in enumerate(pd.read_csv(csv_path,chunksize=chunksize)):\n        result=predict_frame(pipeline,config,frame);result[\'input_row\']+=offset;offset+=len(frame)\n        result.to_csv(output,mode=\'w\' if i==0 else \'a\',header=i==0,index=False)\n    return offset\n'
(OUT/'recall_inference.py').write_text(INFERENCE_MODULE,encoding='utf-8')
module_spec=importlib.util.spec_from_file_location('recall_inference',OUT/'recall_inference.py')
inference=importlib.util.module_from_spec(module_spec);sys.modules['recall_inference']=inference;module_spec.loader.exec_module(inference)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
bundles=OUT/'handoff';bundles.mkdir(exist_ok=True)
for entry in FROZEN['candidates']:
 folder=bundles/entry['trial_id'];folder.mkdir(exist_ok=True)
 assert sha(OUT/entry['model_path'])==entry['model_sha']
 model=joblib.load(OUT/entry['model_path']);names=[FEATURES[i] for i in entry['spec']['features']]
 medians=SC['imputation']['medians']
 pre=ColumnTransformer([(f'f{i}',SimpleImputer(strategy='constant',fill_value=float(medians[f]),keep_empty_features=True),[f]) for i,f in enumerate(names)],remainder='drop',n_jobs=1,verbose_feature_names_out=False)
 pre.fit(pd.DataFrame([{f:float(medians[f]) for f in names}]))
 pipeline=Pipeline([('preprocessing',pre),('float32',FunctionTransformer(np.asarray,kw_args={'dtype':np.float32},validate=False)),
  ('model',inference.DecisionRuleClassifier(model,tuple(entry['rule'])))])
 # Fixed tune samples; confirmation/test do not determine golden sample selection.
 golden_idx=np.concatenate([np.flatnonzero(YTUNE==c)[:2] for c in IDS])
 frame=pd.DataFrame(np.array(XTUNE[golden_idx],dtype=np.float64),columns=FEATURES)
 expected=decide(model.predict_proba(np.ascontiguousarray(frame[names],dtype=np.float32)),entry['rule'])
 assert np.array_equal(pipeline.predict(frame),expected)
 joblib.dump(pipeline,folder/'pipeline.joblib',compress=3)
 del pipeline;gc.collect();loaded=joblib.load(folder/'pipeline.joblib')
 assert np.array_equal(loaded.predict(frame),expected)
 cfg=dict(run_id=RUN_ID,trial_id=entry['trial_id'],primary=entry['trial_id']==FROZEN['primary_trial_id'],
  feature_names=names,class_names=CLASS_NAMES,spec=entry['spec'],multipliers=entry['rule'],
  probabilities='original estimator probabilities; adjusted decision scores not calibrated probabilities',
  frozen_sha=sha(frozen_file),versions=VERSIONS,pipeline_sha=sha(folder/'pipeline.joblib'),
  preprocessing='fixed medians from full source train; no statistical refit')
 write_json(folder/'config.json',cfg);frame[names].to_csv(folder/'golden_features.csv',index=False)
 np.savez_compressed(folder/'golden_expected.npz',prediction=expected,probabilities=loaded.predict_proba(frame),processed_val_rows=VKEEP[TUNE[golden_idx]])
 shutil.copy2(OUT/'recall_inference.py',folder/'recall_inference.py')
 shutil.copy2(OUT/entry['model_path'],folder/'model.joblib')
 # Exported CSV helper is checked inside Kaggle, never locally.
 actual=inference.predict_frame(loaded,cfg,frame)
 assert np.array_equal(actual.predicted_class_id.to_numpy(),expected)
 # Exercise CSV parsing, chunk offsets and loading the serialized bundle.
 n=inference.predict_csv(folder,folder/'golden_features.csv',folder/'golden_predictions.csv',chunksize=7)
 csv_check=pd.read_csv(folder/'golden_predictions.csv')
 assert n==len(frame) and np.array_equal(csv_check.input_row.to_numpy(),np.arange(len(frame)))
 assert np.array_equal(csv_check.predicted_class_id.to_numpy(),expected)
 assert np.allclose(csv_check[['probability_'+c for c in CLASS_NAMES]].to_numpy(),loaded.predict_proba(frame),atol=1e-7,rtol=1e-6)
 write_json(folder/'checks.json',dict(pipeline_reload=True,csv_roundtrip=True,chunk_size=7,samples=n))
 del model,loaded;gc.collect()
print('Handoff:',bundles)


## 12. Tổng hợp, tiêu chí nghiệm thu và dữ liệu bổ sung
Chỉ primary đã khóa được dùng để kết luận; secondary là đối chứng, không đổi primary theo confirmation/test. Target_met phải được đọc riêng trên tune/confirmation; không suy từ điểm tìm kiếm.

Trong tối đa 3 ngày, nhóm khảo sát thêm CSV CICIoT2023 chưa dùng. Nếu có: tạo source manifest/checksum và provenance; kiểm tra schema/mapping; nhóm theo fingerprint raw/float32 và loại nhóm đã có trong cả source train/val/test cũ; kiểm tra nhãn mâu thuẫn; giữ phần mới để đánh giá sau khóa, không dùng điểm mới chọn model. Công bố support, cách lấy mẫu và giới hạn cùng thiết bị/phiên. Nếu không có nguồn đủ điều kiện, không gọi test cũ là test mới và không trì hoãn bàn giao. Việc thu thập nguồn mới là công việc cần thực hiện riêng, không tự tải dữ liệu chưa xác minh.

Ngân sách tính giờ phiên CPU; một fit đang chạy có thể vượt phần còn lại, notebook không hủy giữa fit để tránh artifact hỏng. Final reporting được phép hoàn thành sau khi dừng tìm kiếm và được ghi riêng trong ledger. Không mở tìm kiếm mới sau freeze.


In [ ]:
records=[]
for r in RESULTS.values():
 records.append(dict(id=r['id'],stage=r['stage'],model=r['spec']['model'],sample=r['spec']['sample'],weight=r['spec']['weight'],
  seed=r['spec']['seed'],features=len(r['spec']['features']),rule_BruteForce=r['rule'][0],rule_Web_Based=r['rule'][1],
  fit_seconds=r['fit_seconds'],**{'before_'+k:v for k,v in r['unadjusted_tune_metrics'].items()},**r['tune_metrics']))
pd.DataFrame(records).to_csv(OUT/'ablation_summary.csv',index=False)
primary=FROZEN['primary_trial_id'];confirmation_path=OUT/'assessments'/'confirmation'/primary/'result.json'
confirmation=read_json(confirmation_path) if confirmation_path.exists() else None
status='complete' if confirmation else 'search_complete_confirmation_pending'
ledger=read_json(OUT/'budget_ledger.json') if (OUT/'budget_ledger.json').exists() else []
write_json(OUT/'run_status.json',dict(run_id=RUN_ID,status=status,finished_at=now(),protocol_hash=PH,
 primary_trial_id=primary,confirmation_target_met=confirmation['metrics']['target_met'] if confirmation else None,
 precision_floor=MIN_PRECISION,target_recall=TARGET_RECALL,actual_seed_counts={r['trial_id']:len(r['seeds']) for r in FROZEN['candidates']},
 search_budget_stopped=BUDGET_STOPPED,cumulative_work_hours=sum(r['wall_seconds'] for r in ledger)/3600))
README=f"""# Recall study — {RUN_ID}

Primary (selected by tune only): {primary}. Precision floor: 30% each BruteForce/Web-Based. Recall target: 85% each, not guaranteed.
Status: {status}. Baseline run: {BASELINE_ID}. Validation/test previously viewed; grouped tune/confirm used to control this new search only.

Read search_results.csv / ablation_summary.csv, sampling/*, feature_selection/*, frozen_candidates.json, confirmation_summary.csv and old_test_summary.csv where present. Per-class metrics, recall CI95, confusion matrices, probability rows and sample IDs are in assessments/<split>/<trial>. Read target_met on confirmation, not only tune.

Pipeline:
```python
from recall_inference import predict_csv
predict_csv('handoff/{primary}', 'input.csv', 'predictions.csv')
```
Import recall_inference before joblib.load pipeline. predict uses tuned decision; predict_proba returns original model probabilities. Class order stays 0..7. For SHAP explain underlying estimator model.base_estimator on raw float32 selected feature order; report decision multipliers separately.

Use requirements_runtime.txt and protocol.json to reproduce. RESUME_FROM imports entire earlier output; source/version/protocol must match. Sampling/partition indices, trial checksums and frozen model hashes are saved. Golden samples exercise saved pipeline/CSV helper inside Kaggle. This is not an independent reproduction from raw CSV on another environment.

Limitations: finite rare support, dependence among duplicate/session samples, full-source-train preprocessing reused, validation/test previously viewed, feature projection overlap may reject SHAP subsets, precision/recall may vary across seeds. No fabricated target improvement. Do not reopen searches after confirmation to get prettier points. New unused data requires provenance and overlap audit; no fresh test acquired automatically.

Tasks: Dương owns sample/weight/tuning/decision and model artifacts; Hải owns SHAP/LIME analysis and interpreting raw output vs decision; Kiên owns integration, CSV checking, end-to-end benchmark and guidance. Six-to-eight-week reporting/demo follows the plan.
"""
(OUT/'README_HANDOFF.md').write_text(README,encoding='utf-8')
write_json(OUT/'artifact_checksums.json',{str(p.relative_to(OUT)):sha(p) for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='artifact_checksums.json'})
print('Output:',OUT)
print('Primary:',primary,'| status:',status)
if confirmation:print('Confirmation metrics:',confirmation['metrics'])
print('Tải toàn bộ thư mục run; chỉ tick các bước đã có kết quả thực.')
